# 003 · for Loops and range

> **Syntax › Control Flow** · Python 3.14

Python has exactly one kind of `for` loop, and it is C#'s `foreach`. The classic counting loop `for (int i = 0; i < n; i++)` is expressed by iterating over a `range`.

**In this notebook you will learn:**

- why Python's `for` is always a *foreach*
- `range(start, stop, step)` as the replacement for `for (;;)`
- `enumerate`, `zip` (including `strict=True`) and `reversed`
- iterating dictionaries with `.items()`, `.keys()` and `.values()`
- why you must never modify a list while iterating it
- `_` as a throwaway loop variable

## C# mental model

| C# | Python |
|---|---|
| `foreach (var x in items)` | `for x in items:` |
| `for (int i = 0; i < 5; i++)` | `for i in range(5):` |
| `for (int i = 10; i > 0; i -= 2)` | `for i in range(10, 0, -2):` |
| `items.Select((x, i) => ...)` | `for i, x in enumerate(items):` |
| `a.Zip(b)` | `zip(a, b)` |
| `items.Reverse()` (LINQ) | `reversed(items)` |
| `foreach (var (k, v) in dict)` | `for k, v in d.items():` |

## `for` is always foreach

`for` takes any **iterable** — list, tuple, string, dict, set, file, generator… *(see Iterators & Generators › 001 · Iterables and Iterators)*

```csharp
// C#
foreach (var fruit in fruits)
    Console.WriteLine(fruit);
```

In [ ]:
fruits = ["apple", "banana", "cherry"]
for fruit in fruits:
    print(fruit)

for ch in "hey":                      # strings are iterable (one-character strings)
    print(ch, end=" ")
print()

⚠️ **Gotcha:** the loop variable is **not** scoped to the loop. Unlike C#, it still exists after the loop and holds the last value. (If the iterable was empty, it is never assigned at all.)

In [ ]:
for n in [10, 20, 30]:
    pass
print("n after loop:", n)             # 30 — in C# this would not compile

## `range(start, stop, step)`

`range` produces a lazy sequence of integers. Like slicing, `stop` is **exclusive**:

| Call | Values | C# equivalent |
|---|---|---|
| `range(5)` | 0 1 2 3 4 | `for (int i = 0; i < 5; i++)` |
| `range(2, 6)` | 2 3 4 5 | `for (int i = 2; i < 6; i++)` |
| `range(0, 10, 3)` | 0 3 6 9 | `for (int i = 0; i < 10; i += 3)` |
| `range(5, 0, -1)` | 5 4 3 2 1 | `for (int i = 5; i > 0; i--)` |

C#'s `Enumerable.Range(start, count)` takes a *count*; Python's `range` takes a *stop*.

In [ ]:
print(list(range(5)))
print(list(range(2, 6)))
print(list(range(0, 10, 3)))
print(list(range(5, 0, -1)))
print(list(range(5, 0)))              # empty! a negative step is needed to count down

`range` is lazy and cheap — it doesn't build a list, supports `len`, indexing and fast `in` checks:

In [ ]:
r = range(0, 1_000_000_000, 2)
print(len(r), r[10], 999_999_998 in r)   # instant — no billion-element list is created

⚠️ **Gotcha:** don't write `for i in range(len(items))` just to index into the list — that's a C# habit. Iterate directly, or use `enumerate` if you need the index.

In [ ]:
items = ["a", "b", "c"]

for i in range(len(items)):           # works, but un-Pythonic
    print(i, items[i])

for i, item in enumerate(items):      # idiomatic
    print(i, item)

## `enumerate` — index + item

`enumerate(iterable, start=0)` yields `(index, item)` tuples, which you unpack in the `for` header.

```csharp
// C#
foreach (var (item, i) in items.Select((x, i) => (x, i)))
    Console.WriteLine($"{i}. {item}");
// .NET 9+: foreach (var (i, item) in items.Index())
```

In [ ]:
tasks = ["write tests", "fix bug", "deploy"]
for number, task in enumerate(tasks, start=1):
    print(f"{number}. {task}")

## `zip` — iterate several sequences in lockstep

`zip` pairs up items from several iterables (C#'s LINQ `Zip`). By default it **silently stops at the shortest** input; pass `strict=True` (3.10+) to raise `ValueError` if the lengths differ.

```csharp
// C#
foreach (var (name, score) in names.Zip(scores))
    Console.WriteLine($"{name}: {score}");
```

In [ ]:
names = ["alice", "bob", "carol"]
scores = [91, 78, 85]

for name, score in zip(names, scores):
    print(f"{name}: {score}")

ages = [30, 25, 41]
for name, score, age in zip(names, scores, ages):   # any number of iterables
    print(name, score, age)

In [ ]:
short = [1, 2]
print(list(zip(names, short)))        # carol silently dropped!

try:
    list(zip(names, short, strict=True))
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

## `reversed` — iterate backwards

`reversed(seq)` returns a lazy reverse iterator without copying or modifying the original (unlike `list.reverse()`, which mutates).

```csharp
// C#
for (int i = items.Count - 1; i >= 0; i--) Console.WriteLine(items[i]);
// or: foreach (var x in Enumerable.Reverse(items))
```

In [ ]:
countdown = [1, 2, 3]
for n in reversed(countdown):
    print(n, end=" ")
print("liftoff!", countdown)          # original unchanged

for i, ch in enumerate(reversed("abc")):
    print(i, ch)

## Iterating dictionaries

Iterating a dict directly gives its **keys**. Use `.items()` for key/value pairs (C#'s `KeyValuePair`), or `.values()` for values only. Dicts preserve **insertion order** *(see Collections › 003 · Dictionaries)*.

```csharp
// C#
foreach (var (product, price) in prices)
    Console.WriteLine($"{product}: {price}");
```

In [ ]:
prices = {"apple": 0.5, "bread": 2.25, "milk": 1.1}

for product in prices:                # keys
    print(product, end=" ")
print()

for product, price in prices.items(): # key/value pairs, unpacked
    print(f"{product}: {price:.2f}")

print(sum(prices.values()))

## ⚠️ Never modify a collection while iterating it

In C#, changing a `List<T>` during `foreach` throws `InvalidOperationException`. Python's list doesn't detect this — it **silently skips elements**. Dicts and sets do raise a `RuntimeError` if their size changes.

In [ ]:
numbers = [1, 2, 2, 3, 4]
for n in numbers:
    if n % 2 == 0:
        numbers.remove(n)             # shifts later items left -> the next one is skipped
print(numbers)                        # [1, 2, 3] — one 2 survived!

In [ ]:
stock = {"apple": 0, "bread": 3, "milk": 0}
try:
    for item, qty in stock.items():
        if qty == 0:
            del stock[item]
except RuntimeError as e:
    print(f"{type(e).__name__}: {e}")

**Fixes:** build a new collection (usually a comprehension — *see Collections › 007 · Comprehensions*), or iterate over a **copy**.

In [ ]:
numbers = [1, 2, 2, 3, 4]
numbers = [n for n in numbers if n % 2 != 0]       # build a new list
print(numbers)

stock = {"apple": 0, "bread": 3, "milk": 0}
for item, qty in list(stock.items()):              # iterate over a snapshot
    if qty == 0:
        del stock[item]
print(stock)

## `_` as a throwaway variable

When you don't need the loop variable, name it `_` by convention (C#'s discard `_`). It's also used to ignore parts of an unpacked tuple.

In [ ]:
for _ in range(3):
    print("hello")

points = [(1, 2, 3), (4, 5, 6)]
for x, _, z in points:                # ignore the middle value
    print(x, z)

## Nested loops and unpacking

Nested loops work as you'd expect. Unpacking in the `for` header works with any iterable of fixed-size items *(see Collections › 006 · Unpacking)*.

In [ ]:
for row in range(1, 4):
    line = ""
    for col in range(1, 4):
        line += f"{row * col:4}"
    print(line)

## Summary

| Task | C# | Python |
|---|---|---|
| Foreach | `foreach (var x in xs)` | `for x in xs:` |
| Counting loop | `for (int i = 0; i < n; i++)` | `for i in range(n):` |
| Custom step | `i += 3` | `range(start, stop, 3)` |
| Count down | `i--` | `range(n - 1, -1, -1)` or `reversed(range(n))` |
| Index + item | `.Select((x, i) => ...)` / `.Index()` | `enumerate(xs)` |
| Parallel iteration | `a.Zip(b)` | `zip(a, b, strict=True)` |
| Reverse | `Enumerable.Reverse(xs)` | `reversed(xs)` |
| Dict pairs | `foreach (var (k, v) in d)` | `for k, v in d.items():` |
| Discard | `_` | `_` |
| Remove while iterating | throws | silently wrong — build a new list |

## Exercises

1. Print the even numbers from 20 down to 0 (inclusive) using a single `range`.
2. Given `products = ["pen", "pad", "ink"]` and `prices = [1.5, 3.0, 7.25]`, print a numbered price list starting at 1 (`1. pen — 1.50`). Make it fail loudly if the lists have different lengths.
3. Given `inventory = {"pen": 0, "pad": 4, "ink": 0}`, remove items with zero quantity — without a `RuntimeError`.
4. Why does `for i in range(len(xs)): print(xs[i])` count as a code smell in Python?

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
for n in range(20, -1, -2):
    print(n)

# 2
products = ["pen", "pad", "ink"]
prices = [1.5, 3.0, 7.25]
for i, (product, price) in enumerate(zip(products, prices, strict=True), start=1):
    print(f"{i}. {product} — {price:.2f}")

# 3 — rebuild the dict (or iterate over list(inventory.items()))
inventory = {"pen": 0, "pad": 4, "ink": 0}
inventory = {k: v for k, v in inventory.items() if v > 0}

# 4 — Python's for is a foreach: iterate the items directly
#     (for x in xs) and use enumerate() when you need the index.
```

</details>